In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [3]:


train = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/train.csv',  )


/var/folders/5p/tsf09yfn1d7ct362cxy57hyc0000gn/T/ipykernel_15541/1308030334.py:1: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/train.csv',  )


### I saved a sample of the data to a parquet file so we can all run this as a sanity check, before we scale up


In [ ]:
saved = train.sample(n=1_000_000, random_state=42)
saved.to_parquet('../data/sample/train_sample.parquet')

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# df = train.copy()
# df = train.sample(n=500_000, random_state=42)  # or 1_000_000

df = pd.read_parquet('../data/sample/train_sample.parquet')
# still sort by date before TimeSeriesSplit
# df = df.sort_values("date")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")  # important for TimeSeriesSplit

# numeric date features (keep calendar signal)
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek

# optional: treat onpromotion as 0/1 instead of dropping
df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)

feature_cols = [
    "store_nbr", "item_nbr",
    "year", "month", "day", "dayofweek",
    "onpromotion",
]
X = df[feature_cols]          # no raw date, no id
y = df["unit_sales"]

tscv = TimeSeriesSplit(n_splits=3)
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    model = RandomForestRegressor(n_estimators=20, random_state=42, n_jobs=-1, max_depth=10)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    print(f"Fold {i+1}: {mean_squared_error(y_te, y_pred)}")
    

Fold 1: 290.0085563694221
Fold 2: 1056.6480636122897
Fold 3: 387.5502599226638


In [6]:
from sklearn.metrics import r2_score
# print r2 error
print(f"R2 error: {r2_score(y_te, y_pred)}")

R2 error: 0.12450426537219494


## Practical workflow

| Stage | Data | Goal |
|---|---|---|
| Debug pipeline | 100k–500k rows, 1–3 folds | Does code run? |
| Tune features | ~1–2M rows | MSE improving? |
| Final model | more data / full (or smarter sampling) | submission |